In [11]:
import re
from pathlib import Path
import pandas as pd

In [12]:
# Configuration: folder with the .txt files and folder for the generated .csv files
INPUT_DIR = Path(".")
OUTPUT_DIR = Path("./results/")
PATTERN = "*.txt"

In [13]:
# Regex matching one result line, e.g.:
# Description: [a = '0', b = '3'], Target: c = '1' ; Quality Measure BinomialTest = 48.59 ; tp = 5765 ; fp = 1782 ; TP = 34979 ; FP = 35021
LINE_RE = re.compile(
    r"^Description:\s*\[(?P<Description>.*)\],\s*Target:\s*(?P<Target>.*?)\s*;"
    r"\s*Quality Measure\s+(?P<Measure>\w+)\s*=\s*(?P<Quality>[-+\d.eE]+)\s*;"
    r"\s*tp\s*=\s*(?P<tp>\d+)\s*;"
    r"\s*fp\s*=\s*(?P<fp>\d+)\s*;"
    r"\s*TP\s*=\s*(?P<TP>\d+)\s*;"
    r"\s*FP\s*=\s*(?P<FP>\d+)\s*$"
)

In [14]:
def parse_file(path):
    """Parse a txt file and return a DataFrame sorted by Quality (descending).

    The index keeps the original line order of the file (0-based).
    """
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # skip empty lines
            match = LINE_RE.match(line)
            if match is None:
                print(f"Warning: skipping unparsed line in {path.name}: {line[:80]}")
                continue
            d = match.groupdict()
            rows.append({
                "Description": d["Description"],
                "Quality": float(d["Quality"]),
                "tp": int(d["tp"]),
                "fp": int(d["fp"]),
                "TP": int(d["TP"]),
                "FP": int(d["FP"]),
            })

    df = pd.DataFrame(rows, columns=["Description", "Quality", "tp", "fp", "TP", "FP"])
    # Stable sort so ties keep their original order
    return df.sort_values("Quality", ascending=False, kind="stable")

In [15]:
# Convert every matching txt file into a csv named [algorithm]_[dataset].csv
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for txt_path in sorted(INPUT_DIR.glob(PATTERN)):
    # Expected name: results_[algorithm]_[id]_[dataset].txt
    # maxsplit=3 keeps any underscores in the dataset name intact
    parts = txt_path.stem.split("_", 3)
    if len(parts) != 4 or parts[0] != "results":
        print(f"Warning: skipping {txt_path.name}, unexpected file name")
        continue
    _, algorithm, _, dataset = parts

    df = parse_file(txt_path)
    csv_path = OUTPUT_DIR / f"{algorithm}_{dataset}.csv"
    df.to_csv(csv_path)  # the index is written as an unnamed first column
    print(f"{txt_path.name} -> {csv_path.name} ({len(df)} rows)")

results_BerryFinder_35018_Depression.txt -> BerryFinder_Depression.csv (0 rows)
results_BerryFinder_3909_Mushrooms.txt -> BerryFinder_Mushrooms.csv (0 rows)
results_BerryFinder_49651_Vancomycin.txt -> BerryFinder_Vancomycin.csv (0 rows)
results_BerryFinder_57971_Cardio.txt -> BerryFinder_Cardio.csv (0 rows)
results_BerryFinder_62772_StudentSuccess.txt -> BerryFinder_StudentSuccess.csv (0 rows)
results_BerryFinder_64164_Car.txt -> BerryFinder_Car.csv (0 rows)
results_BerryFinder_72300_connect4.txt -> BerryFinder_connect4.csv (0 rows)
results_BerryFinder_74196_MIMIC-IV.txt -> BerryFinder_MIMIC-IV.csv (0 rows)
results_BerryFinder_75864_Chess.txt -> BerryFinder_Chess.csv (0 rows)
results_BerryFinder_95805_Dermatology.txt -> BerryFinder_Dermatology.csv (0 rows)
